# Program 6: Time Series Forecasting using LSTM

**Course:** Deep Learning Lab — 7th Semester, CSE (AI & ML)  
**Environment:** Google Colab (Runtime → Change runtime type → GPU recommended for CNN / transfer learning)

This notebook follows Experiment 6 in the uploaded *DL Lab Manual*. Run cells from top to bottom. The program saves generated artifacts in `/content/DL_Lab_Outputs/Program_6`.

**Note:** Training accuracy, test accuracy, loss values and predictions can vary between runs and hardware. The notebook generates the actual output files when executed in Colab.

In [ ]:
# Import the package or class needed in this experiment.
import numpy as np
# Import the package or class needed in this experiment.
import pandas as pd
# Import the package or class needed in this experiment.
import tensorflow as tf
# Import the package or class needed in this experiment.
import matplotlib.pyplot as plt
# Import the package or class needed in this experiment.
from pathlib import Path
# Import the package or class needed in this experiment.
from sklearn.preprocessing import MinMaxScaler
# Import the package or class needed in this experiment.
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Import the package or class needed in this experiment.
from tensorflow.keras.models import Sequential
# Import the package or class needed in this experiment.
from tensorflow.keras.layers import LSTM, Dense

# Perform this processing step in the experiment.
tf.keras.utils.set_random_seed(42)
# Assign the resulting value to `rng`.
rng = np.random.default_rng(42)
# Create or define the folder used to store program outputs.
OUT = Path("/content/DL_Lab_Outputs/Program_6")
# Create or define the folder used to store program outputs.
OUT.mkdir(parents=True, exist_ok=True)

# Generate synthetic sinusoidal time-series data
# Assign the resulting value to `time`.
time = np.arange(0, 200, 0.1)
# Assign the resulting value to `data`.
data = np.sin(time) + 0.1 * rng.normal(size=len(time))
# Create, format, save, or display the visualization.
plt.figure(figsize=(10,3)); plt.plot(time, data)
# Create, format, save, or display the visualization.
plt.title("Synthetic Time Series Data"); plt.xlabel("Time"); plt.ylabel("Value")
# Create, format, save, or display the visualization.
plt.tight_layout(); plt.savefig(OUT/"time_series_data.png", dpi=150); plt.show()

# Assign the resulting value to `scaler`.
scaler = MinMaxScaler(feature_range=(0,1))
# Assign the resulting value to `data_scaled`.
data_scaled = scaler.fit_transform(data.reshape(-1,1))

# Define a helper function for the experiment.
def create_sequences(dataset, seq_length):
    # Assign the resulting value to `X, y`.
    X, y = [], []
    # Control execution using this condition, loop, or context.
    for i in range(len(dataset)-seq_length):
        # Perform this processing step in the experiment.
        X.append(dataset[i:i+seq_length, 0])
        # Perform this processing step in the experiment.
        y.append(dataset[i+seq_length, 0])
    # Return the result produced by this function.
    return np.array(X), np.array(y)

# Assign the resulting value to `SEQ_LEN`.
SEQ_LEN = 20
# Assign the resulting value to `X, y`.
X, y = create_sequences(data_scaled, SEQ_LEN)
# Assign the resulting value to `X`.
X = X.reshape((X.shape[0], X.shape[1], 1))
# Assign the resulting value to `split`.
split = int(len(X)*0.8)
# Assign the resulting value to `X_train, X_test`.
X_train, X_test = X[:split], X[split:]
# Assign the resulting value to `y_train, y_test`.
y_train, y_test = y[:split], y[split:]

# Assign the resulting value to `model`.
model = Sequential([LSTM(50, activation="tanh", input_shape=(SEQ_LEN,1)), Dense(1)])
# Configure the optimizer, loss function, and evaluation metrics.
model.compile(optimizer="adam", loss="mse")
# Train the model using the prepared training data.
history = model.fit(X_train, y_train, epochs=10, batch_size=32,
                    # Assign the resulting value to `validation_data`.
                    validation_data=(X_test,y_test), verbose=1)

# Generate predictions for the supplied input samples.
predictions_scaled = model.predict(X_test, verbose=0)
# Assign the resulting value to `predictions`.
predictions = scaler.inverse_transform(predictions_scaled).ravel()
# Assign the resulting value to `actual`.
actual = scaler.inverse_transform(y_test.reshape(-1,1)).ravel()
# Display the current result or evaluation metric.
print("Test MAE:", mean_absolute_error(actual, predictions))
# Display the current result or evaluation metric.
print("Test RMSE:", mean_squared_error(actual, predictions)**0.5)

# Create, format, save, or display the visualization.
plt.figure(); plt.plot(history.history["loss"], label="Train loss")
# Create, format, save, or display the visualization.
plt.plot(history.history["val_loss"], label="Validation loss")
# Create, format, save, or display the visualization.
plt.title("LSTM Training Loss"); plt.xlabel("Epoch"); plt.ylabel("MSE")
# Create, format, save, or display the visualization.
plt.legend(); plt.tight_layout(); plt.savefig(OUT/"training_loss.png", dpi=150); plt.show()
# Create, format, save, or display the visualization.
plt.figure(figsize=(10,4)); plt.plot(actual, label="Actual"); plt.plot(predictions, label="Predicted")
# Create, format, save, or display the visualization.
plt.title("Actual vs Predicted"); plt.legend(); plt.tight_layout()
# Create, format, save, or display the visualization.
plt.savefig(OUT/"actual_vs_predicted.png", dpi=150); plt.show()
# Save experiment results to an output file.
pd.DataFrame({"actual":actual, "predicted":predictions}).to_csv(OUT/"forecast_results.csv", index=False)
# Save experiment results to an output file.
pd.DataFrame(history.history).to_csv(OUT/"training_history.csv", index=False)
# Save the trained model or reusable objects.
model.save(OUT/"lstm_forecaster.keras")